# Tutorial 16: Working with the Future Bank

Every async operation in LAILA returns a future. Every future is registered in the active policy's `future_bank` keyed by its `global_id`. This tutorial shows how to inspect, wait on, and recover errors from futures.

You will:

- Walk `policy.future_bank` after submitting work
- Use `laila.runtime.status`, `laila.runtime.wait`, and `.exception` to inspect a future
- Deliberately raise inside a `ComplexConstitution` and recover the error
- Iterate the child futures of a `GroupFuture`

**Prerequisites:** `pip install laila-core`.

In [ ]:
import laila
from laila.macros.defaults import DefaultPool

laila.memory.extend(DefaultPool(), pool_nickname="bank")

## Submit a batch and inspect the bank

`memorize` on a list returns a `GroupFuture`. Both the group and each child end up in `future_bank`.

In [ ]:
entries = [laila.constant(data=i, nickname=f"bank_{i}") for i in range(5)]
group = laila.memorize(entries, pool_nickname="bank")

bank = laila.active_policy.future_bank
print("bank size:", len(bank))
print("group gid:", group.global_id)
print("first 3 gids:", list(bank.keys())[:3])

## Looking up a future by gid

`laila.runtime` is the canonical way to inspect futures. It accepts a future object, an identity object, or just the `global_id` string.

In [ ]:
group.wait()
print("group status:", laila.runtime.status(group))
print("group result type:", type(laila.runtime.result(group)).__name__)

first_gid = list(bank.keys())[0]
print("status by gid:", laila.runtime.status(first_gid))

## Iterating a GroupFuture's children

A `GroupFuture` carries the `global_id` of each child future on its `future_ids` attribute. Look each one up in the policy's `future_bank` to get the child future and inspect it individually.

In [ ]:
for i, fid in enumerate(group.future_ids):
    child = bank[fid]
    print(f"child {i}: status={laila.runtime.status(child)} gid={fid[:30]}...")

## Recovering from a failed build

Build a `ComplexConstitution` whose code raises. The future captures the exception — `laila.runtime.status` returns `FAILED` and `.exception` yields the original error.

In [ ]:
from laila.entry import Entry
from laila.policy.central.memory.schema.manifest import Manifest

a = laila.constant(data=10)
m = Manifest(data={"a": a}); m.memorize().wait()

BAD_SRC = (
    "def broken(manifest):\n"
    "    raise ValueError('intentional')\n"
)

bad = Entry.variable(constitution=BAD_SRC, manifest=m)
fut = laila.build(bad)
try:
    fut.wait()
except Exception as e:
    print("propagated on wait:", type(e).__name__, "-", e)

print("status:", laila.runtime.status(fut))
print("captured exception:", repr(fut.exception))

## Cancelling pending work at shutdown

`laila.terminate(cancel_pending=True)` drops queued-but-unstarted tasks. Already-running tasks still finish (or fail) — only the unstarted backlog is discarded.

In [ ]:
errors = laila.terminate(wait=True, cancel_pending=True)
print("teardown errors:", errors)

## Summary

- `laila.active_policy.future_bank` is a live dict of every future the policy owns.
- `laila.runtime.status / wait / result` accept futures, identity objects, or gid strings.
- A `GroupFuture` exposes its children for individual inspection.
- Failed futures keep their exception around — read `.exception` instead of expecting `wait()` to swallow it.
- `terminate(cancel_pending=True)` drops queued submissions on the way down.